# Handwritten Character Recognition with a CNN
In this notebook we teach a computer to read handwritten digits. The steps:
1. Load and explore the data  2. Prepare it  3. Build a CNN  4. Train  5. Evaluate  6. Predict your own image  7. Extend to letters

The reusable code lives in `src/`; this notebook explains and runs it step by step.
**Outputs are empty until you run the notebook yourself.**

## 0. Setup
Run this from the repo. In **Google Colab**, first run in a cell:
`!git clone https://github.com/<your-username>/handwritten-character-recognition-cnn.git` then
`%cd handwritten-character-recognition-cnn`, and `!pip install -q -r requirements.txt` if something is missing.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():      # notebook opened from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

## 1. Imports and a random seed
A *seed* makes random choices (weight start values, data split) repeatable, so you and I get the same results.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf

from src.preprocess import (SEED, load_dataset, split_and_normalize, get_class_names,
                            preprocess_custom_image)
from src.model import build_cnn
from src.evaluate import (plot_history, evaluate_predictions, plot_misclassified, top_confusions)

tf.keras.utils.set_random_seed(SEED)
DATASET = "mnist"            # change to "emnist" later
CLASS_NAMES = get_class_names(DATASET)
MODEL_PATH = ROOT / "models" / f"{DATASET}_cnn.keras"
print("TensorFlow", tf.__version__)

## 2. Load the dataset
MNIST has 70,000 pictures of digits written by people. Each picture is a grid of 28 x 28 pixels, and each pixel is a number from 0 (black) to 255 (white). Each picture comes with a **label**: the digit it shows.

In [ ]:
x_train_raw, y_train_raw, x_test_raw, y_test_raw = load_dataset(DATASET)
print("train images:", x_train_raw.shape, x_train_raw.dtype)   # (N, 28, 28)
print("test images :", x_test_raw.shape)
print("pixel range :", x_train_raw.min(), "to", x_train_raw.max())
print("example labels:", y_train_raw[:10])

## 3. Look at the data
Always look at your data before training. Check the images look upright and the classes are roughly balanced.

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for ax, img, lab in zip(axes.ravel(), x_train_raw, y_train_raw):
    ax.imshow(img, cmap="gray"); ax.set_title(CLASS_NAMES[lab]); ax.axis("off")
plt.suptitle("Sample training images"); plt.show()

counts = pd.Series(y_train_raw).map(lambda i: CLASS_NAMES[i]).value_counts().sort_index()
sns.barplot(x=counts.index, y=counts.values); plt.title("Images per class (training set)"); plt.show()

## 4. Prepare the data
- **Normalise**: divide by 255 so pixels are 0-1 (neural networks train better on small numbers).
- **Add a channel axis**: shape `(N, 28, 28, 1)`, the "1" means grayscale.
- **Validation set**: 10% of the *training* data, used to watch for overfitting while training.
- **Test set**: kept untouched until the end. Using it earlier would be *data leakage* and make results look better than they are.

In [ ]:
data = split_and_normalize(x_train_raw, y_train_raw, x_test_raw, y_test_raw)
for name, arr in data.items():
    print(f"{name:8s}", arr.shape)

## 5. Build the CNN
- **Conv2D + ReLU**: small filters slide over the image and light up where they see edges, strokes or curves. ReLU keeps only positive signals.
- **MaxPooling2D**: shrinks the image by keeping the strongest signal in each 2x2 block (less computation, more tolerance to small shifts).
- **Dropout**: randomly switches off some neurons during training so the network cannot just memorise the training images.
- **Flatten**: lays the feature maps out as one long list of numbers.
- **Dense**: ordinary fully connected layers that combine the features into a decision.
- **Softmax output**: one probability per class; they add up to 1.

In [ ]:
model = build_cnn(num_classes=len(CLASS_NAMES))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train
- **Adam** adjusts the network's weights efficiently. **Sparse categorical cross-entropy** measures how wrong the probabilities are (it accepts integer labels).
- **EarlyStopping** stops when validation loss stops improving; **ModelCheckpoint** saves the best version to `models/`.

In [ ]:
MODEL_PATH.parent.mkdir(exist_ok=True)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True),
]
history = model.fit(data["x_train"], data["y_train"],
                    validation_data=(data["x_val"], data["y_val"]),
                    epochs=20, batch_size=128, callbacks=callbacks)

## 7. Learning curves
If training accuracy keeps rising while validation accuracy stalls or drops, the model is **overfitting**.

In [ ]:
plot_history(history.history, show=True)

## 8. Evaluate on the test set
We reload the saved model from disk, which also proves saving and loading works.

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
test_loss, test_acc = model.evaluate(data["x_test"], data["y_test"], verbose=0)
print(f"Test loss: {test_loss:.4f}   Test accuracy: {test_acc:.4f}")

y_pred = model.predict(data["x_test"], verbose=0).argmax(axis=1)
metrics = evaluate_predictions(data["y_test"], y_pred, CLASS_NAMES, show=True)
metrics

**Precision** = of everything predicted as "7", how much really was a 7. **Recall** = of all real 7s, how many we found. **F1** combines both.

In [ ]:
plot_misclassified(data["x_test"], data["y_test"], y_pred, CLASS_NAMES, show=True)
top_confusions(data["y_test"], y_pred, CLASS_NAMES)

### Your observations
Look at the table above and write 2-3 sentences here: which pairs are confused most often, and why might they look alike? (Write only what you see in *your* run.)

## 9. Predict your own handwriting
Our pipeline turns a photo into MNIST format: grayscale -> flip colours if the background is light -> threshold and crop to the character -> resize to fit 20x20 -> centre in 28x28 -> scale to 0-1.
Put your image in `samples/` (dark ink on plain paper works best) and set the filename below.

In [ ]:
from src.predict import predict_image

IMAGE = ROOT / "samples" / "my_digit.png"      # <- change me
result = predict_image(IMAGE, model, CLASS_NAMES, show=True)
print(result)

In [ ]:
# See each preprocessing step
_, steps = preprocess_custom_image(IMAGE, return_steps=True)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (name, img) in zip(axes, steps.items()):
    ax.imshow(img, cmap="gray"); ax.set_title(name); ax.axis("off")
plt.show()

**Limitations**: one character per image; photos with shadows, very thick pens or styles unlike MNIST can fail; a high confidence does not guarantee a correct answer. Test several of your own samples and note which work.

## 10. Extending to letters (EMNIST) - not run here
1. Set `DATASET = "emnist"` in the imports cell (needs `pip install tensorflow-datasets`).
2. Re-run from step 2. The model automatically gets 47 outputs (digits, uppercase, 11 lowercase letters).
3. **Check the sample images in step 3 are upright.** If not, the orientation fix in `load_dataset` needs adjusting.
4. Training is slower (about 112,800 training images); a GPU in Colab helps.